# Imports

In [1]:
import ast
import yaml
import json
import requests
from collections import defaultdict
from lxml import etree
from elasticsearch import Elasticsearch, helpers
from bs4 import BeautifulSoup

# Constants

In [2]:
ES_HOST = "https://prj-ext-prod-planet-bio-dr.es.europe-west2.gcp.elastic-cloud.com"
ES_USERNAME = "elastic"
ES_PASSWORD = "jb0uPcKp4xBSVLSfmvc3H7OA"

# Importing Annotations

In [3]:
annotations = defaultdict(list)
count = 0
for filename in  ["darwin_tree_of_life", "erga_bge", "erga_pilot", "asg", "canadian_biogenome", "vgp"]:
    with open(f'/Users/alexey/ebi_projects/projects.ensembl.org/_data/{filename}/species.yaml', 'r') as yaml_file:
        yaml_data = yaml.safe_load(yaml_file)
        for record in yaml_data:
            annotation = dict()
            annotation['species'] = record['species']
            annotation['accession'] = record['accession']
            count += 1
            print(f"{count}\r", end='', flush=True)
            acc_response = requests.get(f"https://www.ebi.ac.uk/ena/browser/api/xml/{annotation['accession']}")
            try:
                root = etree.fromstring(acc_response.content)
            except etree.XMLSyntaxError:
                print(f"XMLSyntaxError: {annotation['accession']}")
                continue
            try:
                tax_id = root.find("ASSEMBLY").find("TAXON").find("TAXON_ID").text
            except AttributeError:
                if annotation['accession'] == "GCF_902459465.1":
                    tax_id = "7604"
                elif annotation['accession'] == "GCF_902652985.1":
                    tax_id = "6579"
            annotation['tax_id'] = tax_id
            try:
                annotation['annotation'] = {'GTF': record['annotation_gtf'], "GFF3": record['annotation_gff3']}
            except KeyError:
                annotation['annotation'] = {'GTF': None, "GFF3": None}
            try:
                annotation['proteins'] = {'FASTA': record['proteins']}
            except KeyError:
                annotation['proteins'] = {'FASTA': None}
            try:
                annotation['transcripts'] = {'FASTA': record['transcripts']}
            except KeyError:
                annotation['transcripts'] = {'FASTA': None}
            try:
                annotation['softmasked_genome'] = {'FASTA': record['softmasked_genome']}
            except KeyError:
                annotation['softmasked_genome'] = {'FASTA': None}
            try:
                annotation['repeat_library'] = {'FASTA': record['repeat_library']}
            except KeyError:
                annotation['repeat_library'] = None
            annotation['other_data'] = {'ftp_dumps': record['ftp_dumps']}
            try:
                annotation['view_in_browser'] = record['beta_link']
            except KeyError:
                try:
                    annotation['view_in_browser'] = record['ensembl_link']
                except KeyError:
                    annotation['view_in_browser'] = None
            annotations[annotation["tax_id"]].append(annotation)
len(annotations)

1237

1184

In [3]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for tax_id, annotation in annotations.items():
    body = dict()
    body['annotations'] = annotation
    es.index(index="annotation", body=body, id=tax_id)

NameError: name 'annotations' is not defined

# Importing Data Portal metadata

In [28]:
data = list()
with open("/Users/alexey/gbdp_data_portal.txt", "r") as f:
    for line in f:
        line = line.rstrip()
        data.append(ast.literal_eval(line))
len(data)

10665

In [4]:
# To remove duplicated records
for record in data:
    visited_ids = {}
    new_records = {}
    ranks = {
        "Submitted to BioSamples": 1,
        "Raw Data - Submitted": 2,
        "Assemblies - Submitted": 3
    }
    for sample in record["records"]:
        if sample["accession"] not in visited_ids:
            visited_ids[sample["accession"]] = ranks[sample["trackingSystem"]]
            new_records[sample["accession"]] = sample
        else:
            if ranks[sample["trackingSystem"]] > visited_ids[sample["accession"]]:
                visited_ids[sample["accession"]] = ranks[sample["trackingSystem"]]
                new_records[sample["accession"]] = sample
    record["records"] = list(new_records.values())

In [5]:
# To remove duplicated runs
for record in data:
    visited_ids = set()
    new_runs = list()
    for exp in record["experiment"]:
        if exp["run_accession"] not in visited_ids:
            visited_ids.add(exp["run_accession"])
            new_runs.append(exp)
    record["experiment"] = new_runs

In [6]:
# To remove duplicated assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["assemblies"] = new_assemblies

In [7]:
# To remove duplicated symbionts assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["symbionts_assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["symbionts_assemblies"] = new_assemblies

In [8]:
# To remove duplicated metagenomes assemblies
for record in data:
    visited_ids = set()
    new_assemblies = list()
    for assmbl in record["metagenomes_assemblies"]:
        if assmbl["accession"] not in visited_ids:
            visited_ids.add(assmbl["accession"])
            new_assemblies.append(assmbl)
    record["metagenomes_assemblies"] = new_assemblies

In [8]:
data[0].keys()

dict_keys(['tax_id', 'commonName', 'currentStatus', 'experiment', 'assemblies', 'analyses', 'project_name', 'records', 'taxonomies', 'organism', 'commonNameSource', 'symbionts_experiment', 'symbionts_assemblies', 'symbionts_analyses', 'symbionts_records', 'metagenomes_experiment', 'metagenomes_assemblies', 'metagenomes_analyses', 'metagenomes_records', 'annotation', 'biosamples', 'annotation_status', 'annotation_complete', 'assemblies_status', 'mapped_reads', 'raw_data', 'trackingSystem', 'tolid', 'orgGeoList', 'specGeoList', 'genome_notes', 'goat_info', 'nbnatlas', 'show_tolqc', 'tolqc_links'])

In [58]:
actions = list()
for record in data:
    record_to_write = {k: v for k, v in record.items() if k not in  ["analyses", "metagenomes_analyses"]}
    action = {
            "_index": "2025-02-06_data_portal",
            "_id": record_to_write["tax_id"],
            "_source": record_to_write
        }
    actions.append(action)

In [59]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(actions), 1000):
    print(f"Working on {i}: {i+1000}")
    helpers.bulk(es, actions, stats_only=True)

Working on 0: 1000
Working on 1000: 2000
Working on 2000: 3000
Working on 3000: 4000
Working on 4000: 5000
Working on 5000: 6000
Working on 6000: 7000
Working on 7000: 8000
Working on 8000: 9000
Working on 9000: 10000
Working on 10000: 11000


In [52]:
for record in data:
    if record["tax_id"] == 5965:
        print("metagenomes_analyses" in record)
        # record_to_write = {k: v for k, v in record.items() if k != "metagenomes_analyses"}
        # es.index("2025-02-06_data_portal", record_to_write, id=record_to_write["tax_id"])

True


In [42]:
record.keys()

dict_keys(['tax_id', 'commonName', 'currentStatus', 'experiment', 'assemblies', 'analyses', 'project_name', 'records', 'taxonomies', 'organism', 'commonNameSource', 'symbionts_experiment', 'symbionts_assemblies', 'symbionts_analyses', 'symbionts_records', 'metagenomes_experiment', 'metagenomes_assemblies', 'metagenomes_analyses', 'metagenomes_records', 'biosamples', 'annotation_status', 'annotation_complete', 'assemblies_status', 'mapped_reads', 'raw_data', 'trackingSystem', 'tolid', 'orgGeoList', 'specGeoList', 'genome_notes', 'goat_info', 'nbnatlas', 'show_tolqc', 'tolqc_links'])

In [22]:
data_ids = list()
for record in data:
    data_ids.append(record['tax_id'])

In [27]:
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [15]:
data_portal = get_samples("data_portal_new", es)

In [24]:
data_portal = get_samples("data_portal_new", es)
for tax_id in data_portal:
    if int(tax_id) not in data_ids:
        print(tax_id)
        # es.delete("data_portal_new", id=tax_id)

634664
876063_3126489


In [31]:
es.delete("data_portal_test_new", id="634664")

{'_index': 'data_portal_test_new',
 '_id': '634664',
 '_version': 2,
 'result': 'deleted',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 20273,
 '_primary_term': 1}

In [26]:
es.delete("tracking_status_index", id="634664")

{'_index': 'tracking_status_index',
 '_id': '634664',
 '_version': 8,
 'result': 'deleted',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 781770,
 '_primary_term': 11}

In [25]:
data_portal = get_samples("tracking_status_index", es)
for tax_id in data_portal:
    if int(tax_id) not in data_ids:
        es.delete("tracking_status_index", id=tax_id)

In [99]:
es.delete("tracking_status_index", id="13068")

{'_index': 'tracking_status_index',
 '_id': '13068',
 '_version': 7,
 'result': 'deleted',
 '_shards': {'total': 2, 'successful': 1, 'failed': 0},
 '_seq_no': 771577,
 '_primary_term': 11}

# Importing Specimens

In [13]:
specimens = list()
with open("/Users/alexey/specimens.jsonl", "r") as f:
    for line in f:
        line = line.rstrip()
        data_record = ast.literal_eval(line)
        specimens.append({"index": {"_index": "organism", "_id": data_record['accession']}})
        specimens.append(data_record)
len(specimens)

181538

In [14]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(specimens), 10000):
    print(f"Working on {i}: {i+10000}")
    _ = es.bulk(body=specimens[i:i+10000])

Working on 0: 10000
Working on 10000: 20000
Working on 20000: 30000
Working on 30000: 40000
Working on 40000: 50000
Working on 50000: 60000
Working on 60000: 70000
Working on 70000: 80000
Working on 80000: 90000
Working on 90000: 100000
Working on 100000: 110000
Working on 110000: 120000
Working on 120000: 130000
Working on 130000: 140000
Working on 140000: 150000
Working on 150000: 160000
Working on 160000: 170000
Working on 170000: 180000
Working on 180000: 190000


In [32]:
es.indices.get_alias(index="*")

/Users/alexey/.pyenv/versions/3.12.6/lib/python3.12/site-packages/elasticsearch/connection/base.py:208: ElasticsearchWarning: this request accesses system indices: [.apm-custom-link, .tasks, .fleet-policies-7, .fleet-servers-7, .security-tokens-7, .security-7, .kibana_security_session_1, .security-profile-8, .kibana_8.5.0_001, .fleet-enrollment-api-keys-7, .apm-agent-configuration, .fleet-agents-7, .fleet-policies-leader-7, .kibana_task_manager_8.5.0_001], but in a future major version, direct access to system indices will be prevented by default
  warnings.warn(message, category=ElasticsearchWarning)


{'.ent-search-actastic-workplace_search_accounts_v16': {'aliases': {}},
 '.ent-search-actastic-crawler2_configurations': {'aliases': {}},
 '.ent-search-actastic-workplace_search_search_groups_v4-name-unique-constraint': {'aliases': {}},
 '.ent-search-actastic-crawler2_robots_txts': {'aliases': {}},
 '.ent-search-actastic-workplace_search_pre_content_sources_v3': {'aliases': {}},
 '.ent-search-actastic-crawler_crawl_requests_v7': {'aliases': {}},
 'gis_filter_index': {'aliases': {}},
 '.ent-search-esqueues-me_queue_v1_process_crawl2': {'aliases': {}},
 '.ent-search-actastic-reindex_jobs_v3': {'aliases': {}},
 '.ent-search-actastic-workplace_search_role_mappings_v8': {'aliases': {}},
 'organisms_test': {'aliases': {}},
 '.ent-search-actastic-search_relevance_suggestion_update_process_v1': {'aliases': {}},
 '.apm-custom-link': {'aliases': {}},
 '.ent-search-actastic-connectors_jobs_v5': {'aliases': {}},
 'test-20240710-1242': {'aliases': {}},
 '.ent-search-actastic-workplace_search_conten

In [19]:
test1 = {'a': ['a', 'b', 'c'], 'd': ['d', 'e', 'f']}
test2 = {'a': ['a1', 'b1', 'c1'], 'd': ['d1', 'e1', 'f1']}

In [20]:
accumulators = [test1, test2]

In [22]:
tuple(zip(*accumulators))

(('a', 'a'), ('d', 'd'))

In [34]:
def test(*items):
    print(items)

In [35]:
test(*accumulators)

({'a': ['a', 'b', 'c'], 'd': ['d', 'e', 'f']}, {'a': ['a1', 'b1', 'c1'], 'd': ['d1', 'e1', 'f1']})


In [36]:
test = []


In [38]:
test += [1, 2, 3]

In [39]:
test

[1, 2, 3]

In [25]:
actions = []
for tax_id, record in data_portal.items():
    if tax_id == "876063_3126489":
        record["tax_id"] = "3126489"
    action = {
        "_index": "data_portal_new",
        "_id": record["tax_id"],
        "_source": record
        }
    actions.append(action)
    

In [26]:
for i in range(0, len(actions), 1000):
    print(f"Working on {i}: {i+1000}")
    helpers.bulk(es, actions, stats_only=True)

Working on 0: 1000
Working on 1000: 2000
Working on 2000: 3000
Working on 3000: 4000
Working on 4000: 5000
Working on 5000: 6000
Working on 6000: 7000
Working on 7000: 8000
Working on 8000: 9000
Working on 9000: 10000
Working on 10000: 11000


In [27]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

data_portal = get_samples("data_portal_new", es)

RequestError: RequestError(400, 'search_phase_execution_exception', 'Result window is too large, from + size must be less than or equal to: [10000] but was [11000]. See the scroll api for a more efficient way to request large data sets. This limit can be set by changing the [index.max_result_window] index level setting.')

In [28]:
def check_raw_data_status(record):
    if 'experiment' in record and len(record['experiment']) > 0:
        return 'Done'
    else:
        return 'Waiting'


def check_assemblies(record):
    if 'assemblies' in record and len(record['assemblies']) > 0:
        return 'Done'
    else:
        return 'Waiting'


def check_annotation_complete(record):
    if record['currentStatus'] == 'Annotation Complete':
        return 'Done'
    else:
        return 'Waiting'

In [30]:
for organism, record in data_portal.items():
    tmp = dict()
    tmp['organism'] = record['organism']
    tmp['commonName'] = record['commonName']
    tmp['biosamples'] = 'Done'
    tmp['biosamples_date'] = None
    tmp['ena_date'] = None
    tmp['annotation_date'] = None
    tmp['raw_data'] = check_raw_data_status(record)
    tmp['mapped_reads'] = tmp['raw_data']
    tmp['assemblies'] = check_assemblies(record)
    tmp['annotation'] = 'Waiting'
    tmp['annotation_complete'] = check_annotation_complete(record)
    tmp['trackingSystem'] = [
        {'name': 'biosamples', 'status': 'Done', 'rank': 1},
        {'name': 'mapped_reads', 'status': tmp['mapped_reads'], 'rank': 2},
        {'name': 'assemblies', 'status': tmp['assemblies'], 'rank': 3},
        {'name': 'raw_data', 'status': tmp['raw_data'], 'rank': 4},
        {'name': 'annotation', 'status': 'Waiting', 'rank': 5},
        {'name': 'annotation_complete', 'status': tmp['annotation_complete'], 'rank': 6}
    ]
    if 'taxonomies' in record:
        tmp['taxonomies'] = record['taxonomies']
    if 'symbionts_records' in record:
        tmp['symbionts_records'] = record['symbionts_records']
    if 'symbionts_assemblies' in record:
        tmp['symbionts_assemblies'] = record['symbionts_assemblies']
    if 'symbionts_status' in record:
        tmp['symbionts_status'] = record['symbionts_status']
    if 'symbionts_experiments' in record:
        tmp['symbionts_experiments'] = record['symbionts_experiments']
    if 'symbionts_biosamples_status' in record:
        tmp['symbionts_biosamples_status'] = record['symbionts_biosamples_status']
    if 'symbionts_assemblies_status' in record:
        tmp['symbionts_assemblies_status'] = record['symbionts_assemblies_status']
    if 'metagenomes_records' in record:
        tmp['metagenomes_records'] = record['metagenomes_records']
    if 'metagenomes_experiments' in record:
        tmp['metagenomes_experiments'] = record['metagenomes_experiments']
    if 'metagenomes_assemblies' in record:
        tmp['metagenomes_assemblies'] = record['metagenomes_assemblies']
    if 'metagenomes_biosamples_status' in record:
        tmp['metagenomes_biosamples_status'] = record['metagenomes_biosamples_status']
    if 'metagenomes_assemblies_status' in record:
        tmp['metagenomes_assemblies_status'] = record['metagenomes_assemblies_status']
    es.index("tracking_status_index", tmp, id=organism)

In [7]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [61]:
data_portal = get_samples("data_portal_new", es)

RequestError: RequestError(400, 'search_phase_execution_exception', 'Result window is too large, from + size must be less than or equal to: [10000] but was [11000]. See the scroll api for a more efficient way to request large data sets. This limit can be set by changing the [index.max_result_window] index level setting.')

In [ ]:
tracking_status = get_samples("tracking_status_index_test", es)

In [4]:
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [14]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
data_portal = get_samples("2025-02-18_data_portal", es)

In [7]:
articles = list()
for tax_id, record in data_portal.items():
    print(f"{list(data_portal.keys()).index(tax_id)/len(data_portal)*100}\r", end='', flush=True)
    if 'genome_notes' in record and len(record["genome_notes"]) > 0:
        for article in record["genome_notes"]:
            article_response = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={article['study_id']}&format=json").json()
            if len(article_response['resultList']['result']) > 0:
                pub_year = article_response['resultList']['result'][0]['pubYear']
                article['pub_year'] = pub_year
                article['pubYear'] = pub_year
            else:
                article['pub_year'] = None
                article['pubYear'] = None
            article['id'] = article['study_id']
            article['articleType'] = 'Genome Note'
            article['journalTitle'] = 'Wellcome Open Res'
            article['organism_name'] = record['organism']
            articles.append({"index": {"_index": "articles", "_id": article['study_id']}})
            articles.append(article)

99.99031289353879654

In [8]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(articles), 10000):
    print(f"Working on {i}: {i+10000}")
    _ = es.bulk(body=articles[i:i+10000])

Working on 0: 10000


In [9]:
list(data_portal.keys())[0]

'116153'

In [12]:
data_portal["572874"]["annotation"]

[{'species': 'Pasiphila rectangulata',
  'accession': 'GCA_963082625.1',
  'tax_id': '572874',
  'annotation': {'GTF': 'https://ftp.ensembl.org/pub/rapid-release/species/Pasiphila_rectangulata/GCA_963082625.1/braker/geneset/2023_10/Pasiphila_rectangulata-GCA_963082625.1-2023_10-genes.gtf.gz',
   'GFF3': 'https://ftp.ensembl.org/pub/rapid-release/species/Pasiphila_rectangulata/GCA_963082625.1/braker/geneset/2023_10/Pasiphila_rectangulata-GCA_963082625.1-2023_10-genes.gff3.gz'},
  'proteins': {'FASTA': 'https://ftp.ensembl.org/pub/rapid-release/species/Pasiphila_rectangulata/GCA_963082625.1/braker/geneset/2023_10/Pasiphila_rectangulata-GCA_963082625.1-2023_10-pep.fa.gz'},
  'transcripts': {'FASTA': 'https://ftp.ensembl.org/pub/rapid-release/species/Pasiphila_rectangulata/GCA_963082625.1/braker/geneset/2023_10/Pasiphila_rectangulata-GCA_963082625.1-2023_10-cdna.fa.gz'},
  'softmasked_genome': {'FASTA': 'https://ftp.ensembl.org/pub/rapid-release/species/Pasiphila_rectangulata/GCA_963082625

In [7]:
data_portal = get_samples("data_portal", es)

In [8]:
len(data_portal)

10569

In [ ]:
for tax_id, record in data_portal.items():
    if ''

In [11]:
for tax_id, record in data_portal.items():
    if "AEGIS (Ancient Environmental Genomics Initiative for Sustainability)" in record["project_name"]:
        print(tax_id)

37682
4565
4571


In [3]:
data = list()
with open("/Users/alexey/aegis/data_portal.jsonl", "r") as f:
    for line in f:
        line = line.rstrip()
        data.append(ast.literal_eval(line))
len(data)

72

In [4]:
data[0].keys()

dict_keys(['tax_id', 'commonName', 'currentStatus', 'experiment', 'assemblies', 'analyses', 'project_name', 'records', 'taxonomies', 'organism', 'commonNameSource', 'symbionts_experiment', 'symbionts_assemblies', 'symbionts_analyses', 'symbionts_records', 'metagenomes_experiment', 'metagenomes_assemblies', 'metagenomes_analyses', 'metagenomes_records', 'biosamples', 'annotation_status', 'annotation_complete', 'assemblies_status', 'mapped_reads', 'raw_data', 'trackingSystem', 'tolid', 'orgGeoList', 'specGeoList', 'genome_notes', 'goat_info', 'show_tolqc', 'tolqc_links'])

In [6]:
es = Elasticsearch(["https://prj-ext-prod-aegis-dp-452010.es.europe-west2.gcp.elastic-cloud.com"], http_auth=("elastic", "h6hut9kpF2MXhIzXXk7LUmeA"))
phylogenetic_ranks = ('kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species')
for record in data:
    aegis_record = {}
    print(record["tax_id"])
    aegis_record["taxId"] = record["tax_id"]
    aegis_record["scientificName"] = record["organism"]
    aegis_record["commonName"] = record["commonName"]

    aegis_record["phylogeny"] = {}
    for rank_name, rank_value in record["taxonomies"].items():
        if rank_name in phylogenetic_ranks:
            aegis_record["phylogeny"][rank_name] = rank_value["scientificName"]

    aegis_record["samples"] = []
    for sample in record["records"]:
        sample_record = {sample_field_name: sample_field_value for sample_field_name, sample_field_value in sample.items()}
        sample_record["scientificName"] = sample_record["organism"]["text"]
        del sample_record["organism"]
        aegis_record["samples"].append(sample_record)
    
    aegis_record["currentStatus"] = record["currentStatus"]
    aegis_record["bioSamplesStatus"] = record["biosamples"]
    aegis_record["rawDataStatus"] = record["raw_data"]
    aegis_record["assembliesStatus"] = record["assemblies_status"]
    if aegis_record["currentStatus"] == "Assemblies - Submitted":
        aegis_record["currentStatusOrder"] = 3
    elif aegis_record["currentStatus"] == "Raw Data - Submitted":
        aegis_record["currentStatusOrder"] = 2
    else:
        aegis_record["currentStatusOrder"] = 1

    aegis_record["rawData"] = record["experiment"]
    aegis_record["assemblies"] = record["assemblies"]

    es.index("2025-03-07_data_portal", aegis_record, id=record["tax_id"])

93721


/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_7283/1613122922.py:36: DeprecationWarning: Using positional arguments for APIs is deprecated and will be disabled in 8.0.0. Instead use only keyword arguments for all APIs. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.index("2025-03-07_data_portal", aegis_record, id=record["tax_id"])


1464905
384672
44658
202321
88556
2818049
500459
1425403
486019
994498
44665
2026903
208863
133901
1425463
1425404
527821
93723
330455
1393966
69196
87316
214836
1393965
527766
44664
223852
50529
345560
3447467
13131
796291
49225
120009
1276742
85692
32644
384675
224526
1336441
1499505
295196
486017
133092
796228
74065
34666
222432
163907
7038
74070
527890
132712
434553
135958
345556
74068
4565
511033
4571
330452
202456
3384248
279269
2898095
24953
2562066
44659
37682
36667
527743


In [24]:
data[2]["tax_id"]

37682

In [17]:
annotations = get_samples("annotation", es)

In [18]:
ES_HOST = "https://prj-ext-prod-erga-gcp-dr.es.europe-west2.gcp.elastic-cloud.com"
ES_USERNAME = "elastic"
ES_PASSWORD = "aCJuMoynlz190jYcw6Kb3gFE"

In [3]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))

In [5]:
def get_samples(index_name, es):
    samples = dict()
    search_body = {"size": 10000, "sort": [{"tax_id": "asc"}]}
    response = es.search(index=index_name, body=search_body)
    while len(response["hits"]["hits"]) != 0:
        for sample in response["hits"]["hits"]:
            samples[sample["_id"]] = sample["_source"]
        search_body["search_after"] = response["hits"]["hits"][-1]["sort"]
        response = es.search(index=index_name, body=search_body)
    return samples

In [6]:
data_portal = get_samples("data_portal", es)

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_66153/1799168123.py:4: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  response = es.search(index=index_name, body=search_body)
/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_66153/1799168123.py:9: DeprecationWarning: The 'body' parameter is deprecated for the 'search' API and will be removed in a future version. Instead use API parameters directly. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  response = es.search(index=index_name, body=search_body)


In [10]:
data_portal["878968"]["genome_notes"]

[{'tax_id': '878968',
  'study_id': 'PRJEB51454',
  'url': 'https://wellcomeopenresearch.org/articles/8-579/v1',
  'citeURL': 'https://doi.org/10.12688/wellcomeopenres.20500.1',
  'title': 'The genome sequence of a longhorn beetle, <i>Rutpela maculata</i> (Poda, 1769)',
  'abstract': '<p>We present a genome assembly from an individual female <i>Rutpela maculata</i> (a longhorn beetle; Arthropoda; Insecta; Coleoptera; Cerambycidae). The genome sequence is 2,021.6 megabases in span. Most of the assembly is scaffolded into 10 chromosomal pseudomolecules, including the X sex chromosome. The mitochondrial genome has also been assembled and is 17.84 kilobases in length. Gene annotation of this assembly on Ensembl identified 33,598 protein coding genes.</p>',
  'figureURI': 'https://wellcomeopenresearch.s3.eu-west-1.amazonaws.com/manuscripts/22693/c1dd225b-da15-499c-95b9-27b1f419c2d1_figure1.gif',
  'caption': 'Figure 1. '}]

In [11]:
for tax_id, record in data_portal.items():
    try:
        if len(record["genome_notes"]) > 1:
            print(tax_id)
    except KeyError:
        continue

43151


In [13]:
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [14]:
articles = get_samples("articles", es)

In [15]:
list(articles.keys())[0]

'PRJEB43008'

In [16]:
data_portal_ids = list()
for tax_id, record in data_portal.items():
    try:
        for article in record["genome_notes"]:
            data_portal_ids.append(article["study_id"])
    except KeyError:
        continue

In [19]:
len(data_portal_ids)

1154

In [18]:
data_portal_ids = list(set(data_portal_ids))

In [22]:
for article_id in list(articles.keys()):
    if article_id not in data_portal_ids:
        es.delete("articles", id=article_id)

/var/folders/71/4c_77lrn65x46bxkhf0k6bj80000gp/T/ipykernel_66153/1909968004.py:3: DeprecationWarning: Using positional arguments for APIs is deprecated and will be disabled in 8.0.0. Instead use only keyword arguments for all APIs. See https://github.com/elastic/elasticsearch-py/issues/1698 for more information
  es.delete("articles", id=article_id)


In [30]:
articles = get_samples("articles", es)

NotFoundError: NotFoundError(404, 'index_not_found_exception', 'no such index [articles]', articles, index_or_alias)

In [31]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
data_portal = get_samples("data_portal", es)

In [32]:
articles = list()
for tax_id, record in data_portal.items():
    print(f"{list(data_portal.keys()).index(tax_id)/len(data_portal)*100}\r", end='', flush=True)
    if 'genome_notes' in record and len(record["genome_notes"]) > 0:
        for article in record["genome_notes"]:
            article_response = requests.get(f"https://www.ebi.ac.uk/europepmc/webservices/rest/search?query={article['study_id']}&format=json").json()
            if len(article_response['resultList']['result']) > 0:
                pub_year = article_response['resultList']['result'][0]['pubYear']
                article['pub_year'] = pub_year
                article['pubYear'] = pub_year
            else:
                article['pub_year'] = None
                article['pubYear'] = None
            article['id'] = article['study_id']
            article['articleType'] = 'Genome Note'
            article['journalTitle'] = 'Wellcome Open Res'
            article['organism_name'] = record['organism']
            articles.append({"index": {"_index": "articles", "_id": article['study_id']}})
            articles.append(article)

99.99087341425573475

In [33]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
for i in range(0, len(articles), 10000):
    print(f"Working on {i}: {i+10000}")
    _ = es.bulk(body=articles[i:i+10000])

Working on 0: 10000


In [3]:
es = Elasticsearch([ES_HOST], http_auth=(ES_USERNAME, ES_PASSWORD))
def get_samples(index_name, es):
    samples = dict()
    data = es.search(index=index_name, size=1000)
    offset = 0
    while len(data['hits']['hits']) > 0:
        for sample in data['hits']['hits']:
            samples[sample['_id']] = sample['_source']
        offset += 1000
        data = es.search(index=index_name, size=1000, from_=offset)
    return samples

In [7]:
data_portal = get_samples("data_portal", es)
images_available = 0
for organism_name, record in data_portal.items():
    if record["images_available"] is True:
        images_available += 1
print(images_available)

2225


In [11]:
output = open("asg_species_groups.jsonl", "w")
with open("/Users/alexey/names.txt", "r") as f:
    for line in f:
        line = line.rstrip()
        data = line.split("\t")
        tmp = {"scientific_name": data[0], "species_group": data[1]}
        output.write(f"{json.dumps(tmp)}\n")
output.close()